## 0. Combine Individual Runtraces into `runtrace_combined_revised.json`

Loads all `runtraces/runtrace_*.json` files, validates each against the ms1 schema, combines them into a single document with correctly computed `aggregate_metrics`, and writes `runtrace_combined_revised.json`.

In [14]:
import json
import re
from pathlib import Path
from dataclasses import dataclass, field
from collections import Counter

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
RUNTRACES_DIR = Path("runtraces")
SCHEMA_PATH = Path("runtrace_ms1.schema.json")
OUTPUT_PATH = Path("runtrace_combined_revised.json")

VALID_LABELS = {"ENTAILED", "CONTRADICTED", "NOT_MENTIONED"}
VALID_SEVERITIES = {"LOW", "MEDIUM", "HIGH"}
VALID_ACTIONS = {"ACCEPT", "CLARIFY", "NEGOTIATE", "ESCALATE"}
VALID_STEP_TYPES = {"load_inputs", "nli_infer", "evidence_map", "playbook_map",
                    "consistency_check", "format_response"}
VALID_COMPONENTS = {"pipeline", "model", "playbook", "validator"}
VALID_CRITICALITIES = {"P0", "P1", "P2"}
VALID_STATUSES = {"PASS", "FAIL", "WARN"}
HYPOTHESIS_IDS = {f"H{i:02d}" for i in range(1, 18)}

# ---------------------------------------------------------------------------
# Report helper
# ---------------------------------------------------------------------------
@dataclass
class ValidationReport:
    errors: list = field(default_factory=list)
    warnings: list = field(default_factory=list)
    info: list = field(default_factory=list)

    def error(self, scope: str, msg: str):
        self.errors.append(f"[ERROR] {scope}: {msg}")

    def warn(self, scope: str, msg: str):
        self.warnings.append(f"[WARN]  {scope}: {msg}")

    def ok(self, scope: str, msg: str):
        self.info.append(f"[OK]    {scope}: {msg}")

    def summary(self):
        status = "PASS" if not self.errors else "FAIL"
        print(f"\n{'='*70}")
        print(f"  VALIDATION {status}  |  {len(self.errors)} errors, "
              f"{len(self.warnings)} warnings, {len(self.info)} passed")
        print(f"{'='*70}")
        for line in self.errors:
            print(line)
        for line in self.warnings:
            print(line)
        if not self.errors and not self.warnings:
            print("  All checks passed.")
        print()

report = ValidationReport()

# ---------------------------------------------------------------------------
# 1. Load all individual runtrace files
# ---------------------------------------------------------------------------
files = sorted(RUNTRACES_DIR.glob("runtrace_*.json"),
               key=lambda f: int(f.stem.split("_")[1]))
print(f"Found {len(files)} runtrace files in {RUNTRACES_DIR}/")

traces = []
for f in files:
    with open(f, "r", encoding="utf-8") as fh:
        traces.append(json.load(fh))

with open(SCHEMA_PATH, "r", encoding="utf-8") as fh:
    schema = json.load(fh)

# ---------------------------------------------------------------------------
# 2. Validate shared top-level fields
# ---------------------------------------------------------------------------
first = traces[0]

sv = first.get("schema_version")
if sv == "1.0-ms1":
    report.ok("schema_version", f"'{sv}'")
else:
    report.error("schema_version", f"Expected '1.0-ms1', got '{sv}'")

run = first.get("run", {})
for req in ["run_id", "started_at", "ended_at", "framework", "parameters"]:
    if req not in run:
        report.error("run", f"Missing required field '{req}'")
if run.get("framework") != "single_model_pipeline":
    report.error("run.framework", f"Expected 'single_model_pipeline', got '{run.get('framework')}'")

params = run.get("parameters", {})
for req in ["base_model", "quantization", "adapter_method", "seed"]:
    if req not in params:
        report.error("run.parameters", f"Missing required field '{req}'")
if params.get("quantization") not in (None, "4bit"):
    report.error("run.parameters.quantization", f"Expected '4bit', got '{params.get('quantization')}'")
if params.get("adapter_method") not in (None, "lora", "qlora"):
    report.error("run.parameters.adapter_method",
                 f"Expected 'lora'|'qlora', got '{params.get('adapter_method')}'")
seed = params.get("seed")
if seed is not None and (not isinstance(seed, int) or seed < 0):
    report.error("run.parameters.seed", f"Expected non-negative integer, got {seed}")
if len(run.get("run_id", "")) < 8:
    report.error("run.run_id", f"Minimum length 8")
report.ok("run", f"run_id={run.get('run_id')}, framework={run.get('framework')}")

pb = first.get("playbook", {})
for req in ["playbook_id", "version", "ruleset_hash"]:
    if req not in pb:
        report.error("playbook", f"Missing required field '{req}'")
rh = pb.get("ruleset_hash", "")
if not re.fullmatch(r"[a-fA-F0-9]{64}", rh):
    report.error("playbook.ruleset_hash", f"Must be 64-char hex")
report.ok("playbook", f"playbook_id={pb.get('playbook_id')}, version={pb.get('version')}")

# ---------------------------------------------------------------------------
# 3. Validate each per-contract runtrace & build combined structure
# ---------------------------------------------------------------------------
contract_traces = []
all_run_validations = []

# Aggregate accumulators
total_hypotheses = 0
total_correct = 0
total_compliant = 0
total_quote_integrity = 0
total_latency = 0.0
agg_label_counts = Counter()
agg_confusion_counts = Counter()
ts_violations = 0
quote_total = 0
quote_found = 0
quote_missing_examples = []

for file_idx, t in enumerate(traces):
    cid = t.get("contract", {}).get("contract_id", f"_file{file_idx}")
    scope_c = f"contract[{cid}]"
    contract = t.get("contract", {})
    metrics = t.get("metrics", {})
    htraces = t.get("hypothesis_traces", [])

    # --- contract object ---
    for req in ["contract_id", "source_type", "hash_sha256", "chunks"]:
        if req not in contract:
            report.error(scope_c, f"Missing required field '{req}'")
    st = contract.get("source_type")
    if st not in ("txt", "pdf", "docx", "unknown"):
        report.error(scope_c, f"Invalid source_type '{st}'")
    h = contract.get("hash_sha256", "")
    if not re.fullmatch(r"[a-fA-F0-9]{64}", h):
        report.error(scope_c, f"hash_sha256 not 64-char hex")

    chunks = contract.get("chunks", [])
    if not isinstance(chunks, list) or len(chunks) == 0:
        report.error(scope_c, "chunks must be a non-empty array")

    chunk_map = {}
    for ci, chunk in enumerate(chunks):
        for req in ["chunk_id", "text", "span"]:
            if req not in chunk:
                report.error(f"{scope_c}.chunk[{ci}]", f"Missing '{req}'")
        span = chunk.get("span", {})
        if "char_start" not in span or "char_end" not in span:
            report.error(f"{scope_c}.chunk[{ci}].span", "Missing char_start/char_end")
        chunk_map[chunk.get("chunk_id", f"_missing_{ci}")] = chunk.get("text", "")

    # --- hypothesis traces ---
    if len(htraces) != 17:
        report.error(scope_c, f"Expected 17 hypothesis_traces, got {len(htraces)}")

    seen_hids = set()
    recomputed_correct = 0
    recomputed_labels = Counter()
    recomputed_confusion = Counter()

    for hi, ht in enumerate(htraces):
        scope_h = f"{scope_c}.hyp[{hi}]"
        for req in ["hypothesis_id", "hypothesis_text", "gold_label", "latency_ms",
                     "compliant_evidence_required", "quote_integrity_pass",
                     "steps", "decision", "validations"]:
            if req not in ht:
                report.error(scope_h, f"Missing required field '{req}'")

        hid = ht.get("hypothesis_id", "???")
        if not re.fullmatch(r"H(0[1-9]|1[0-7])", hid):
            report.error(scope_h, f"hypothesis_id '{hid}' doesn't match H01-H17")
        if hid in seen_hids:
            report.error(scope_c, f"Duplicate hypothesis_id '{hid}'")
        seen_hids.add(hid)

        gold = ht.get("gold_label")
        if gold not in VALID_LABELS:
            report.error(scope_h, f"Invalid gold_label '{gold}'")

        lat = ht.get("latency_ms")
        if lat is not None and (not isinstance(lat, (int, float)) or lat < 0):
            report.error(scope_h, f"latency_ms must be >= 0")

        # steps
        steps = ht.get("steps", [])
        if len(steps) < 1:
            report.error(scope_h, "steps must have at least 1 entry")
        for si, step in enumerate(steps):
            for req in ["step_id", "step_type", "producer", "started_at",
                         "ended_at", "inputs", "outputs"]:
                if req not in step:
                    report.error(f"{scope_h}.steps[{si}]", f"Missing '{req}'")
            if step.get("step_type") not in VALID_STEP_TYPES:
                report.error(f"{scope_h}.steps[{si}]",
                             f"Invalid step_type '{step.get('step_type')}'")
            prod = step.get("producer", {})
            if "component" not in prod:
                report.error(f"{scope_h}.steps[{si}].producer", "Missing 'component'")
            elif prod["component"] not in VALID_COMPONENTS:
                report.error(f"{scope_h}.steps[{si}].producer",
                             f"Invalid component '{prod['component']}'")
            sa, ea = step.get("started_at", ""), step.get("ended_at", "")
            if sa and ea and sa > ea:
                ts_violations += 1
                if ts_violations <= 5:
                    report.error(f"{scope_h}.steps[{si}]",
                                 f"started_at ({sa}) > ended_at ({ea})")
        for si in range(len(steps) - 1):
            ea = steps[si].get("ended_at", "")
            sa_next = steps[si + 1].get("started_at", "")
            if ea and sa_next and ea > sa_next:
                ts_violations += 1
                if ts_violations <= 5:
                    report.warn(f"{scope_h}.steps[{si}->{si+1}]",
                                f"ended_at ({ea}) > next started_at ({sa_next})")

        # decision
        decision = ht.get("decision", {})
        for req in ["label", "confidence", "evidence", "justification", "risk"]:
            if req not in decision:
                report.error(f"{scope_h}.decision", f"Missing '{req}'")

        pred = decision.get("label")
        if pred not in VALID_LABELS:
            report.error(f"{scope_h}.decision", f"Invalid label '{pred}'")

        conf = decision.get("confidence")
        if conf is not None and (not isinstance(conf, (int, float)) or conf < 0 or conf > 1):
            report.error(f"{scope_h}.decision", f"confidence must be 0-1")

        evidence = decision.get("evidence", {})
        if "supporting" not in evidence or "counter" not in evidence:
            report.error(f"{scope_h}.decision.evidence", "Missing 'supporting' or 'counter'")
        else:
            for etype in ("supporting", "counter"):
                raw_items = evidence.get(etype, [])
                # Flatten nested lists: [[{...}]] -> [{...}]
                flat_items = []
                for x in raw_items:
                    if isinstance(x, list):
                        flat_items.extend(x)
                    else:
                        flat_items.append(x)
                for ei, item in enumerate(flat_items):
                    iscope = f"{scope_h}.decision.evidence.{etype}[{ei}]"
                    for req in ["chunk_id", "quote", "relevance_score"]:
                        if req not in item:
                            report.error(iscope, f"Missing '{req}'")
                    ecid = item.get("chunk_id")
                    if ecid and ecid not in chunk_map:
                        report.error(iscope, f"chunk_id '{ecid}' not in contract chunks")
                    rs = item.get("relevance_score")
                    if rs is not None and (not isinstance(rs, (int, float)) or rs < 0 or rs > 1):
                        report.error(iscope, f"relevance_score must be 0-1")
                    # quote integrity
                    quote = item.get("quote", "")
                    chunk_text = chunk_map.get(ecid, "")
                    quote_total += 1
                    if quote and chunk_text:
                        if " ".join(quote.split()) in " ".join(chunk_text.split()):
                            quote_found += 1
                        elif len(quote_missing_examples) < 5:
                            quote_missing_examples.append(
                                f"{scope_c}.{hid}.{etype}[{ei}]: "
                                f"'{quote[:60]}...' not in chunk {ecid}")

        justification = decision.get("justification", {})
        for req in ["claim", "inference", "limitations"]:
            if req not in justification:
                report.error(f"{scope_h}.decision.justification", f"Missing '{req}'")

        risk = decision.get("risk", {})
        for req in ["severity", "playbook_rule_ids", "recommended_action"]:
            if req not in risk:
                report.error(f"{scope_h}.decision.risk", f"Missing '{req}'")
        if risk.get("severity") not in VALID_SEVERITIES and "severity" in risk:
            report.error(f"{scope_h}.decision.risk",
                         f"Invalid severity '{risk.get('severity')}'")
        if risk.get("recommended_action") not in VALID_ACTIONS and "recommended_action" in risk:
            report.error(f"{scope_h}.decision.risk",
                         f"Invalid recommended_action '{risk.get('recommended_action')}'")
        crit = risk.get("criticality")
        if crit is not None and crit not in VALID_CRITICALITIES:
            report.warn(f"{scope_h}.decision.risk",
                        f"criticality '{crit}' not in schema enum {VALID_CRITICALITIES}")

        # per-hypothesis validations
        validations = ht.get("validations", [])
        for vi, v in enumerate(validations):
            for req in ["validator_id", "status", "message"]:
                if req not in v:
                    report.error(f"{scope_h}.validations[{vi}]", f"Missing '{req}'")
            if v.get("status") not in VALID_STATUSES and "status" in v:
                report.error(f"{scope_h}.validations[{vi}]",
                             f"Invalid status '{v.get('status')}'")

        # accumulate recomputed metrics (predicted|gold convention)
        if pred:
            recomputed_labels[pred] += 1
        if gold and pred:
            recomputed_confusion[f"{pred}|{gold}"] += 1
        if gold == pred:
            recomputed_correct += 1

    missing = HYPOTHESIS_IDS - seen_hids
    if missing:
        report.error(scope_c, f"Missing hypothesis_ids: {sorted(missing)}")

    # --- metrics cross-check ---
    hc = metrics.get("hypothesis_count", 0)
    if hc != 17:
        report.error(f"{scope_c}.metrics", f"hypothesis_count expected 17, got {hc}")

    lc = metrics.get("label_counts", {})
    if sum(lc.values()) != hc:
        report.error(f"{scope_c}.metrics", f"label_counts sum != hypothesis_count")

    if metrics.get("correct_count", -1) != recomputed_correct:
        report.error(f"{scope_c}.metrics",
                     f"correct_count={metrics.get('correct_count')} but recomputed={recomputed_correct}")

    for label in VALID_LABELS:
        if lc.get(label, 0) != recomputed_labels.get(label, 0):
            report.error(f"{scope_c}.metrics",
                         f"label_counts[{label}]={lc.get(label, 0)} recomputed={recomputed_labels.get(label, 0)}")

    reported_cc = metrics.get("confusion_counts", {})
    for key in set(list(reported_cc.keys()) + list(recomputed_confusion.keys())):
        if reported_cc.get(key, 0) != recomputed_confusion.get(key, 0):
            report.error(f"{scope_c}.metrics",
                         f"confusion_counts[{key}]={reported_cc.get(key, 0)} recomputed={recomputed_confusion.get(key, 0)}")

    # --- run_validations ---
    rv = t.get("run_validations", [])
    for vi, v in enumerate(rv):
        for req in ["validator_id", "status", "message"]:
            if req not in v:
                report.error(f"{scope_c}.run_validations[{vi}]", f"Missing '{req}'")
    all_run_validations.extend(rv)

    # --- accumulate into combined ---
    contract_traces.append({
        "contract": contract,
        "hypothesis_traces": htraces,
        "contract_metrics": metrics,
    })

    total_hypotheses += hc
    total_correct += recomputed_correct
    total_compliant += metrics.get("compliant_count", 0)
    total_quote_integrity += metrics.get("quote_integrity_count", 0)
    total_latency += metrics.get("contract_latency_ms", 0)
    for label in VALID_LABELS:
        agg_label_counts[label] += recomputed_labels.get(label, 0)
    agg_confusion_counts += recomputed_confusion

# ---------------------------------------------------------------------------
# 4. Timestamp & quote integrity summary
# ---------------------------------------------------------------------------
if ts_violations > 5:
    report.error("timestamps", f"{ts_violations} total ordering violations (showed first 5)")
elif ts_violations == 0:
    report.ok("timestamps", "All step timestamps correctly ordered")

if quote_total > 0:
    qmiss = quote_total - quote_found
    qrate = quote_found / quote_total
    if qmiss > 0:
        report.warn("quote_integrity",
                     f"{quote_found}/{quote_total} quotes found ({qrate:.1%}), {qmiss} missing")
        for ex in quote_missing_examples:
            report.warn("quote_integrity", f"  example: {ex}")
    else:
        report.ok("quote_integrity", f"All {quote_total} quotes found in chunk text")

# ---------------------------------------------------------------------------
# 5. Build aggregate metrics & write combined file
# ---------------------------------------------------------------------------
contract_count = len(traces)
aggregate_metrics = {
    "contract_count": contract_count,
    "hypothesis_count": total_hypotheses,
    "correct_count": total_correct,
    "compliant_count": total_compliant,
    "quote_integrity_count": total_quote_integrity,
    "label_accuracy": round(total_correct / total_hypotheses, 4) if total_hypotheses else 0,
    "groundedness_rate": round(total_compliant / total_hypotheses, 4) if total_hypotheses else 0,
    "quote_integrity_rate": round(total_quote_integrity / total_hypotheses, 4) if total_hypotheses else 0,
    "avg_latency_ms": round(total_latency / contract_count, 2) if contract_count else 0,
    "label_counts": dict(agg_label_counts),
    "confusion_counts": dict(agg_confusion_counts),
}

combined = {
    "schema_version": first["schema_version"],
    "run": run,
    "playbook": dict(pb),
    "aggregate_metrics": aggregate_metrics,
    "contract_traces": contract_traces,
    "run_validations": all_run_validations,
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as fh:
    json.dump(combined, fh, indent=2, ensure_ascii=False)

report.ok("output", f"Written {OUTPUT_PATH} ({OUTPUT_PATH.stat().st_size:,} bytes)")
report.ok("output", f"{contract_count} contracts, {total_hypotheses} hypotheses, "
          f"label_accuracy={aggregate_metrics['label_accuracy']}")

# ---------------------------------------------------------------------------
# 6. Final report
# ---------------------------------------------------------------------------
report.summary()

Found 123 runtrace files in runtraces/

  VALIDATION PASS  |  0 errors, 6 warnings, 6 passed
[WARN]  quote_integrity: 4603/6597 quotes found (69.8%), 1994 missing
[WARN]  quote_integrity:   example: contract[1].H01.supporting[6]: 'g. any computer software, source code, object code, flow cha...' not in chunk 1_chunk_0
[WARN]  quote_integrity:   example: contract[1].H01.supporting[7]: 'h. any personal identifiable information (PII) or other info...' not in chunk 1_chunk_0
[WARN]  quote_integrity:   example: contract[1].H03.supporting[0]: 'Confidential Information means any data or information that ...' not in chunk 1_chunk_0
[WARN]  quote_integrity:   example: contract[1].H03.supporting[5]: 'e. any protected, non-public information concerning the secu...' not in chunk 1_chunk_0
[WARN]  quote_integrity:   example: contract[1].H03.supporting[6]: 'f. any information related to the security of existing or pr...' not in chunk 1_chunk_0

